Movie Recommnedation Based On Previously Watched + Similar Users watched

In [ ]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity


In [ ]:
# sample vector similarity


v1 = [3.5, 2.5, 4]
v2 = [4, 5, 1]
v3 = [2, 1, 4]


X = [v1, v2, v3]

similarity = cosine_similarity(X)
similarity

In [ ]:

movies = pd.read_csv("data/movies.csv")

movies

In [ ]:
ratings = pd.read_csv("data/ratings.csv")

ratings

In [ ]:
movies = movies.head(1000)
ratings = ratings.head(100000)

In [ ]:
ratings

In [ ]:
ratings["userId"].nunique()

In [ ]:
user_data = ratings.pivot(index="userId", columns="movieId", values="rating").fillna(0)
user_data

In [ ]:
user_similarity = cosine_similarity(user_data)
user_similarity

In [ ]:
user_similarity.shape

In [ ]:
user_similarity_df = pd.DataFrame(user_similarity, index=user_data.index, columns=user_data.index)
user_similarity_df

In [ ]:
user_id = 49
user_vector = user_similarity_df[user_id]
user_vector

In [ ]:
user_vector.sort_values(ascending=False)

In [ ]:
user_ratings_row = user_data.loc[user_id]
user_ratings_row

Show all movies this user watched and rated

In [ ]:
for movie_id, rating in user_ratings_row.items():
    if rating > 0:
        print(f"User {user_id} rated movie {movie_id}", "Movie name:", movies[movies["movieId"] == movie_id]["title"].values[0], f"with a rating of {rating}")

Illustration
each movie gets a score  based on similarity with the user and the rating the user has give



In [ ]:
weighted_ratings = pd.Series(dtype=float)

count = 0
for sim_user, similarity_score in user_vector.items():
    count += 1
    # print(f"Sim user: {sim_user}, Similarity score: {similarity_score}")
    
    sim_user_ratings = user_data.loc[sim_user]
    weighted_ratings = weighted_ratings.add(sim_user_ratings * similarity_score, fill_value=0)
    print("count: ", count)

weighted_ratings

In [ ]:
weighted_ratings.sort_values(ascending=False)

In [ ]:
movvie_id =356
movie_name = movies[movies["movieId"] == movvie_id]["title"].values[0]
print(f"Movie ID: {movvie_id}, Movie name: {movie_name}")

In [ ]:
user_rated = user_data.loc[user_id]
user_rated_movies = user_rated[user_rated > 0].index.tolist()
print("User rated movies:", user_rated_movies)

In [ ]:
# Remove movies that the user has watched
weighted_ratings_after_removing_watched = weighted_ratings.drop(index=user_rated_movies, errors='ignore')

weighted_ratings_after_removing_watched

In [ ]:
weighted_ratings_after_removing_watched.sort_values(ascending=False)[:10]

In [1]:
! pip install numpy==1.26.4

In [2]:
! pip install scikit-surprise

In [10]:
import pandas as pd
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise import accuracy


ratings = pd.read_csv("data/ratings.csv")[:1000000]  # Use a subset for faster processing

ratings.head()

,userId,movieId,rating,timestamp
0,1,17,4.0,944249077
1,1,25,1.0,944250228
2,1,29,2.0,943230976
3,1,30,5.0,944249077
4,1,32,5.0,943228858


In [12]:
numbers = [1,2,3]

print(type(numbers))

<class 'list'>


In [11]:

# Define rating scale
reader = Reader(rating_scale=(0.5, 5.0))

# Load data into Surprise
data = Dataset.load_from_df(
    ratings[['userId', 'movieId', 'rating']],
    reader
)

data

In [13]:
print(type(data))

<class 'surprise.dataset.DatasetAutoFolds'>


In [ ]:
# y = mx + c

In LLM case 
x can be the prompt, 
y is the LLM Generated 
m, c are the model weights... 

Ollama model -> oubli => m, c traiined

In [14]:

trainset, testset = train_test_split(
    data,
    test_size=0.2,
    random_state=42
)


In [15]:
trainset

In [ ]:
model = SVD(
    n_factors=100,
    n_epochs=20,
    lr_all=0.005,
    reg_all=0.02,
    random_state=42
)


In [ ]:
model.fit(trainset)

In [18]:
model

In [20]:
predictions = model.test(testset)

In [21]:
len(predictions)

200000

In [52]:
predictions_10 = predictions[0:2]
predictions_10

[Prediction(uid=86, iid=2132, r_ui=5.0, est=4.044287818822157, details={'was_impossible': False}),
 Prediction(uid=2270, iid=7373, r_ui=2.5, est=3.2525081717094153, details={'was_impossible': False})]

Mean Absolute Error (MAE)

In [32]:
d1 = 5.0 - 4.044287818822157
d1

0.9557121811778426

In [33]:
d2 = 2.5 - 3.2525081717094153
d2

-0.7525081717094153

In [35]:
absolutes = 0.9557121811778426 + 0.7525081717094153
absolutes

1.708220352887258

In [37]:
mean_absolutes = absolutes/2
mean_absolutes

0.854110176443629

In [39]:
MAE_10 = accuracy.mae(predictions_10)

MAE:  0.8541


In [40]:
len(predictions)

200000

In [43]:
MAE_200000 = accuracy.mae(predictions)


MAE:  0.6322


In [45]:
u_est  = 3.2525081717094153 + 0.6322
u_est

3.8847081717094154

In [46]:
l_est  = 3.2525081717094153 - 0.6322
l_est

2.620308171709415

In [48]:
2**3

8

In [ ]:
import math
s1 = (5.0 - 4.044287818822157)**2
s2 = (2.5 - 3.2525081717094153)**2

math.sqrt((s1 + s2)/2)

1.2164104248735936

In [59]:
predictions_10

[Prediction(uid=86, iid=2132, r_ui=5.0, est=4.044287818822157, details={'was_impossible': False}),
 Prediction(uid=2270, iid=7373, r_ui=2.5, est=3.2525081717094153, details={'was_impossible': False})]

In [ ]:
MSE = accuracy.rmse(predictions_10)

MSE: 0.7398


In [63]:
model.pu[0]

array([ 0.22870619,  0.14086222, -0.02159205,  0.19644499,  0.15089265,
       -0.09844565,  0.10434846,  0.02525162, -0.09683615, -0.01574273,
        0.14236235,  0.20691445,  0.06215568,  0.05747926, -0.08646432,
       -0.1648687 ,  0.09846098,  0.08745495,  0.19859186,  0.04229576,
        0.32127542, -0.0639862 , -0.19387002, -0.03208123,  0.05235431,
       -0.03786045, -0.04986744, -0.00536937,  0.11886006, -0.15910883,
        0.077287  , -0.06001586,  0.06139588, -0.20335004,  0.01765011,
       -0.02613248,  0.13771143,  0.23509093, -0.11737095, -0.09399132,
        0.13383303,  0.00393184,  0.26508685,  0.00999217, -0.01394914,
       -0.02188828, -0.10758061, -0.08604964, -0.00493706, -0.16539281,
        0.1455924 , -0.06069231,  0.04796567, -0.16325951, -0.04206239,
       -0.03321858, -0.15135256, -0.02370803,  0.04272115,  0.053909  ,
       -0.00868966, -0.32301558,  0.00808724, -0.15880221, -0.00042181,
        0.07082898,  0.22718571, -0.11884205, -0.17815626, -0.23

In [64]:
model.qi[0]

array([ 1.24386828e-02,  3.73539826e-02,  4.44609221e-02,  1.30599056e-01,
        6.26590770e-02,  2.34221479e-03,  1.25012828e-01, -6.60194481e-02,
       -6.80997030e-02,  1.68449688e-01, -5.09589551e-02, -5.68647881e-02,
        1.14438065e-01,  1.08334575e-01, -1.29452788e-01, -2.25145877e-01,
        4.81414081e-02, -8.24766701e-02, -4.23065318e-02,  2.45739113e-01,
        1.97272702e-01,  1.68069958e-01,  7.27584820e-02,  1.11165613e-01,
        1.73035470e-01, -1.24413729e-01, -4.85586090e-02,  1.09528283e-01,
        6.44510985e-02, -8.01031789e-02, -6.36232334e-02,  1.45893236e-02,
        3.65487552e-02, -3.88105671e-02, -1.59381182e-01,  3.00494976e-02,
        1.82750643e-01, -3.05812520e-02,  7.39617739e-02,  2.24989733e-01,
        2.05595260e-02, -1.41674118e-01, -1.07684815e-01,  1.05075599e-01,
        1.25622745e-02, -2.84791222e-02, -5.32987293e-02, -2.54495758e-03,
       -3.27224200e-02,  1.38474170e-01,  1.62000210e-02, -3.78130626e-02,
       -1.20962392e-01, -

In [70]:
predcit_u1_m1 = model.predict(uid=0, iid=0)
predcit_u1_m1

Prediction(uid=0, iid=0, r_ui=None, est=3.56496, details={'was_impossible': False})